# Comparacion de modelos - YOLOv8m-seg V6 vs RT-DETR-L matched

Este notebook valida ambos modelos sobre el mismo split `val` de BlackjackVAI V4 y mide metricas de calidad, coste e inferencia para la tabla final del README.

**Entradas esperadas:**

- `models/best.pt` - YOLOv8m-seg V6 (modelo de producción).
- `models/best_rtdetr_matched.pt` - RT-DETR-L entrenado con config matched (mismo dataset, augmentations, epochs/batch/seed; optimizer y lr0 propios de RT-DETR).
- `BlackjackVAI-4/data.yaml` - mismo dataset/split de entrenamiento.

Las metricas de YOLO se toman en modo bbox (`m.box.*`) para que la comparacion sea 1:1 con RT-DETR.

## 0. Configuracion

In [1]:
from pathlib import Path
import itertools
import json
import random
import statistics
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO, RTDETR
from ultralytics.utils.torch_utils import get_flops

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path(".").resolve()
DATA_YAML = BASE_DIR / "BlackjackVAI-4" / "data.yaml"
VAL_IMG_DIR = BASE_DIR / "BlackjackVAI-4" / "val" / "images"
if not VAL_IMG_DIR.exists():
    VAL_IMG_DIR = BASE_DIR / "BlackjackVAI-4" / "valid" / "images"
YOLO_PATH = BASE_DIR / "models" / "best.pt"
RTDETR_PATH = BASE_DIR / "models" / "best_rtdetr_matched.pt"
OUT_DIR = BASE_DIR / "comparison_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMGSZ = 640
VAL_CONF = 0.001
VAL_IOU = 0.6
PRED_CONF = 0.25
N_BENCHMARK = 200
WARMUP = 20

required = {
    "DATA_YAML": DATA_YAML,
    "VAL_IMG_DIR": VAL_IMG_DIR,
    "YOLO_PATH": YOLO_PATH,
    "RTDETR_PATH": RTDETR_PATH,
}
for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(f"Falta {name}: {path}")

print(f"Dataset : {DATA_YAML}")
print(f"Val imgs: {VAL_IMG_DIR}")
print(f"YOLO    : {YOLO_PATH}")
print(f"RT-DETR : {RTDETR_PATH}")
print(f"Output  : {OUT_DIR}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

Dataset : C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\BlackjackVAI-4\data.yaml
Val imgs: C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\BlackjackVAI-4\val\images
YOLO    : C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\models\best.pt
RT-DETR : C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\models\best_rtdetr_matched.pt
Output  : C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\comparison_runs
CUDA    : True
GPU     : NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Cargar modelos

In [2]:
models = {
    "YOLOv8m-seg V6": {
        "backend": "yolo",
        "path": YOLO_PATH,
        "model": YOLO(str(YOLO_PATH)),
    },
    "RT-DETR-L matched": {
        "backend": "rtdetr",
        "path": RTDETR_PATH,
        "model": RTDETR(str(RTDETR_PATH)),
    },
}

for name, item in models.items():
    print(f"{name:20s} -> {item['path']}")

YOLOv8m-seg V6       -> C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\models\best.pt
RT-DETR-L matched    -> C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\models\best_rtdetr_matched.pt


## 2. Validacion comun sobre split `val`

Se usa `conf=0.001` para mAP y `iou=0.6`, manteniendo el mismo `data.yaml`, `split` e `imgsz`.

In [3]:
def validate_model(name: str, model):
    run_name = name.lower().replace(" ", "_").replace("-", "_")
    metrics = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=IMGSZ,
        conf=VAL_CONF,
        iou=VAL_IOU,
        plots=True,
        save_json=True,
        project=str(OUT_DIR / "val"),
        name=run_name,
        exist_ok=True,
        verbose=False,
    )
    return {
        "mAP50_box": float(metrics.box.map50),
        "mAP50_95_box": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "val_save_dir": str(metrics.save_dir),
    }

val_results = {}
for name, item in models.items():
    print(f"Validando {name}...")
    val_results[name] = validate_model(name, item["model"])

pd.DataFrame(val_results).T

Validando YOLOv8m-seg V6...
Ultralytics 8.4.52  Python-3.11.14 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)


YOLOv8m-seg summary (fused): 106 layers, 27,253,650 parameters, 0 gradients, 104.5 GFLOPs


val: Fast image access  (ping: 0.10.0 ms, read: 43.99.8 MB/s, size: 18.8 KB)


val: Scanning C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\BlackjackVAI-4\valid\labels.cache... 644 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 644/644  0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 2% ──────────── 1/41 3.3s/it 1.0s<2:13

...

                   all        644        718      0.969      0.975       0.98      0.916


Speed: 1.2ms preprocess, 37.0ms inference, 0.0ms loss, 0.4ms postprocess per image


Saving C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\comparison_runs\val\rt_detr_l_matched\predictions.json...


Results saved to C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\comparison_runs\val\rt_detr_l_matched


,mAP50_box,mAP50_95_box,precision,recall,val_save_dir
YOLOv8m-seg V6,0.985957,0.979857,0.976774,0.985461,C:\Users\javie\OneDrive\Escritorio\Master\Visi...
RT-DETR-L matched,0.9804,0.915811,0.968791,0.97516,C:\Users\javie\OneDrive\Escritorio\Master\Visi...


## 3. Coste del modelo: parametros, FLOPs y tamano en disco

In [4]:
def safe_flops(model, imgsz=640):
    try:
        return float(get_flops(model.model, imgsz=imgsz)) / 1e9
    except Exception as exc:
        print(f"FLOPs no disponibles para {type(model).__name__}: {exc}")
        return np.nan

model_stats = {}
for name, item in models.items():
    model = item["model"]
    path = item["path"]
    params_m = sum(p.numel() for p in model.model.parameters()) / 1e6
    model_stats[name] = {
        "params_M": params_m,
        "flops_G_640": safe_flops(model, IMGSZ),
        "size_MB": path.stat().st_size / 1e6,
    }

pd.DataFrame(model_stats).T

,params_M,flops_G_640,size_MB
YOLOv8m-seg V6,27.25365,1.044611e-07,54.930181
RT-DETR-L matched,32.09471,1.036690e-07,66.436285


## 4. Benchmark de latencia y VRAM

Se descartan las primeras `WARMUP` inferencias para calentar GPU. La latencia se mide imagen a imagen sobre hasta 200 imagenes del split de validacion.

In [5]:
def val_image_paths():
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    paths = sorted(p for p in VAL_IMG_DIR.rglob("*.*") if p.suffix.lower() in exts)
    if not paths:
        raise RuntimeError(f"No hay imagenes en {VAL_IMG_DIR}")
    random.shuffle(paths)
    target = min(len(paths), N_BENCHMARK + WARMUP)
    return paths[:target]

bench_paths = val_image_paths()
print(f"Imagenes benchmark: {len(bench_paths)}")


def benchmark_model(model, image_paths):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    timings_ms = []
    repeated = itertools.cycle(image_paths)
    total = max(len(image_paths), WARMUP + 1)

    for i, path in zip(range(total), repeated):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model.predict(str(path), imgsz=IMGSZ, conf=PRED_CONF, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = (time.perf_counter() - t0) * 1000
        if i >= WARMUP:
            timings_ms.append(elapsed)

    peak_vram_mb = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else np.nan
    return {
        "latency_ms_mean": statistics.mean(timings_ms),
        "latency_ms_p50": statistics.median(timings_ms),
        "latency_ms_p95": float(np.percentile(timings_ms, 95)),
        "fps": 1000.0 / statistics.mean(timings_ms),
        "vram_MB": peak_vram_mb,
        "samples": len(timings_ms),
    }

bench_results = {}
for name, item in models.items():
    print(f"Benchmark {name}...")
    bench_results[name] = benchmark_model(item["model"], bench_paths)

pd.DataFrame(bench_results).T

Imagenes benchmark: 220
Benchmark YOLOv8m-seg V6...


Benchmark RT-DETR-L matched...


,latency_ms_mean,latency_ms_p50,latency_ms_p95,fps,vram_MB,samples
YOLOv8m-seg V6,41.830056,41.41635,44.551565,23.906255,673.139712,200.0
RT-DETR-L matched,58.090250,57.78410,61.850890,17.214593,663.079424,200.0


## 5. Tabla resumen y export markdown

In [6]:
rows = []
for name in models:
    row = {"model": name, "backend": models[name]["backend"]}
    row.update(val_results[name])
    row.update(model_stats[name])
    row.update(bench_results[name])
    rows.append(row)

summary = pd.DataFrame(rows).set_index("model")
summary_path_csv = OUT_DIR / "comparison_summary.csv"
summary_path_md = OUT_DIR / "comparison_summary.md"
summary.to_csv(summary_path_csv)
summary.to_markdown(summary_path_md)

print(f"CSV      -> {summary_path_csv}")
print(f"Markdown -> {summary_path_md}")
summary

CSV      -> C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\comparison_runs\comparison_summary.csv
Markdown -> C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\comparison_runs\comparison_summary.md


,backend,mAP50_box,mAP50_95_box,precision,recall,val_save_dir,params_M,flops_G_640,size_MB,latency_ms_mean,latency_ms_p50,latency_ms_p95,fps,vram_MB,samples
model,,,,,,,,,,,,,,,
YOLOv8m-seg V6,yolo,0.985957,0.979857,0.976774,0.985461,C:\Users\javie\OneDrive\Escritorio\Master\Visi...,27.25365,1.044611e-07,54.930181,41.830056,41.41635,44.551565,23.906255,673.139712,200
RT-DETR-L matched,rtdetr,0.980400,0.915811,0.968791,0.975160,C:\Users\javie\OneDrive\Escritorio\Master\Visi...,32.09471,1.036690e-07,66.436285,58.090250,57.78410,61.850890,17.214593,663.079424,200


## 6. Figura de metricas principales

In [7]:
plot_df = summary[["mAP50_box", "mAP50_95_box", "fps", "params_M", "size_MB"]].copy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_df[["mAP50_box", "mAP50_95_box"]].plot(kind="bar", ax=axes[0], rot=15)
axes[0].set_title("Calidad bbox")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("mAP")

plot_df[["fps"]].plot(kind="bar", ax=axes[1], rot=15, legend=False, color="#2f7d55")
axes[1].set_title("FPS efectivo @640")
axes[1].set_ylabel("FPS")

plot_df[["params_M", "size_MB"]].plot(kind="bar", ax=axes[2], rot=15)
axes[2].set_title("Coste")
axes[2].set_ylabel("M params / MB")

for ax in axes:
    ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
fig_path = OUT_DIR / "comparison_metrics.png"
fig.savefig(fig_path, dpi=160, bbox_inches="tight")
print(f"Figura -> {fig_path}")
plt.show()

Figura -> C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\comparison_runs\comparison_metrics.png


<Figure size 1500x400 with 3 Axes>

## 7. Analisis cualitativo lado a lado

Genera una parrilla con las mismas imagenes de validacion procesadas por ambos modelos.

In [8]:
def read_rgb(path: Path):
    img = cv2.imread(str(path))
    if img is None:
        raise RuntimeError(f"No se pudo leer {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def side_by_side_grid(image_paths, title="Comparacion cualitativa"):
    rows = len(image_paths)
    fig, axes = plt.subplots(rows, 3, figsize=(15, 4.5 * rows))
    axes = np.atleast_2d(axes)

    for row, path in enumerate(image_paths):
        axes[row, 0].imshow(read_rgb(path))
        axes[row, 0].set_title(f"Original\n{path.name}", fontsize=8)
        axes[row, 0].axis("off")

        for col, name in enumerate(models, start=1):
            result = models[name]["model"].predict(str(path), imgsz=IMGSZ, conf=PRED_CONF, verbose=False)[0]
            annotated = cv2.cvtColor(result.plot(line_width=2), cv2.COLOR_BGR2RGB)
            dets = len(result.boxes) if result.boxes is not None else 0
            axes[row, col].imshow(annotated)
            axes[row, col].set_title(f"{name}\n{dets} det.", fontsize=8)
            axes[row, col].axis("off")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout()
    return fig

qual_paths = bench_paths[:9]
fig = side_by_side_grid(qual_paths, title="YOLOv8m-seg V6 vs RT-DETR-L matched - muestras VAL")
qual_path = OUT_DIR / "qualitative_grid.png"
fig.savefig(qual_path, dpi=160, bbox_inches="tight")
print(f"Grid -> {qual_path}")
plt.show()

Grid -> C:\Users\javie\OneDrive\Escritorio\Master\Vision Artificial\blackjack-VAI\comparison_runs\qualitative_grid.png


<Figure size 1500x4050 with 27 Axes>

## 8. Cinco casos dificiles seleccionados a mano

Rellena `HARD_CASES` con rutas concretas del split `val` para documentar: carta lejana, carta ocluida, dos cartas solapadas, iluminacion lateral y carta rotada >30 grados.

In [9]:
HARD_CASES = [
    # VAL_IMG_DIR / "nombre_imagen_1.jpg",
    # VAL_IMG_DIR / "nombre_imagen_2.jpg",
]

if HARD_CASES:
    fig = side_by_side_grid(HARD_CASES, title="Casos dificiles - comparacion cualitativa")
    hard_path = OUT_DIR / "hard_cases_grid.png"
    fig.savefig(hard_path, dpi=160, bbox_inches="tight")
    print(f"Casos dificiles -> {hard_path}")
    plt.show()
else:
    print("Añade rutas a HARD_CASES cuando tengas seleccionadas las 5 imagenes dificiles.")

Añade rutas a HARD_CASES cuando tengas seleccionadas las 5 imagenes dificiles.


## 9. Bloque para copiar al README

In [10]:
readme_cols = [
    "mAP50_box", "mAP50_95_box", "precision", "recall",
    "fps", "latency_ms_mean", "params_M", "flops_G_640", "size_MB", "vram_MB",
]
readme_table = summary[readme_cols].round(3)
print(readme_table.to_markdown())

| model             |   mAP50_box |   mAP50_95_box |   precision |   recall |    fps |   latency_ms_mean |   params_M |   flops_G_640 |   size_MB |   vram_MB |
|:------------------|------------:|---------------:|------------:|---------:|-------:|------------------:|-----------:|--------------:|----------:|----------:|
| YOLOv8m-seg V6    |       0.986 |          0.98  |       0.977 |    0.985 | 23.906 |             41.83 |     27.254 |             0 |    54.93  |   673.14  |
| RT-DETR-L matched |       0.98  |          0.916 |       0.969 |    0.975 | 17.215 |             58.09 |     32.095 |             0 |    66.436 |   663.079 |
